In [ ]:
import os
if not os.path.isdir("models"):
    %cd ../
else:
    print("HEHEHE")

In [ ]:
# Do for both: binary + multiclass
classify_type = "binary"

In [ ]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, BaggingClassifier
import lightgbm as lgb
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split,learning_curve
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
import joblib as jl
from tqdm import tqdm
import numpy as np
from torch import nn
from skorch import NeuralNetClassifier
from sklearn.utils import resample
import torch

In [ ]:
datasets = r"../Datasets/ClaMP_In_Selected.csv"
df = pd.read_csv(datasets)
df2 = pd.read_csv(r"../Datasets/ClaMP_In_Selected_Test.csv")

if classify_type == "multiclass":
    df3 = pd.read_csv(r"../Datasets/ClaMP_In_Selected_Additional.csv")
    df = pd.concat([df,df3])
df2 = df2.sample(frac = 1)
# Seperate the data
X_hehe,y = df.drop(columns=["class"]), df["class"].to_numpy(np.float32)
X_hehe_2, y_true = df2.drop(columns=["class"]), df2["class"].to_numpy(np.float32)
for i,label in enumerate(y):
    if label!=0:
        y[i]=1
for i,label in enumerate(y_true):
    if label!=0:
        y_true[i]=1
ohc = OneHotEncoder()
ohc.fit(y.reshape(-1, 1))
y_ohe = ohc.transform(y.reshape(-1, 1)).toarray()
y_true_ohe = ohc.transform(y_true.reshape(-1, 1)).toarray()
out = ["e_lfanew", "NumberOfSections", "BaseOfData", "SizeOfStackReserve", "SizeOfHeapReserve", "SizeOfHeapCommit", "E_text", "E_data", "filesize", "E_file"]
categories = [sorted(list(set(X_hehe[col]).union(set(X_hehe_2[col])))) for col in X_hehe.columns if col not in out]
ohe = OneHotEncoder(categories=categories,handle_unknown="ignore")
scaler = StandardScaler()
categorical_1 = X_hehe.drop(columns = out).to_numpy()
categorical_2 = X_hehe_2.drop(columns = out).to_numpy()
non_categorical_1 = X_hehe[out].to_numpy()
non_categorical_2 = X_hehe_2[out].to_numpy()
scaler.fit(non_categorical_1)
trans_non_cate1 = scaler.transform(non_categorical_1)
trans_non_cate2 = scaler.transform(non_categorical_2)
ohe.fit(categorical_1)
trans_cate_1 = ohe.transform(categorical_1).toarray()
ohe.fit(categorical_2)
trans_cate_2 = ohe.transform(categorical_2).toarray()
X = np.concatenate((trans_cate_1, trans_non_cate1), axis = 1)
X_true_hehe = np.concatenate((trans_cate_2, trans_non_cate2), axis = 1)
X_train, X_test, y_train, y_test = train_test_split(X,y, random_state = 16, test_size = 0.25)
X_train, X_test, y_train_ohe, y_test_ohe = train_test_split(X,y_ohe, random_state = 16, test_size = 0.25)
X_true, X_valid, y_true, y_valid = train_test_split(X_true_hehe,y_true, random_state = 16, test_size = 0.25)
X_true, X_valid, y_true_ohe, y_valid_ohe = train_test_split(X_true_hehe,y_true_ohe, random_state = 16, test_size = 0.25)
X_train = torch.from_numpy(X_train).unsqueeze(1).to(torch.float32)
X_test = torch.from_numpy(X_test).unsqueeze(1).to(torch.float32)
X_true = torch.from_numpy(X_true).unsqueeze(1).to(torch.float32)
X_valid = torch.from_numpy(X_valid).unsqueeze(1).to(torch.float32)
y_train_ohe = torch.from_numpy(y_train_ohe).to(torch.float32)
y_test_ohe = torch.from_numpy(y_test_ohe).to(torch.float32)
y_true_ohe = torch.from_numpy(y_true_ohe).to(torch.float32)
y_valid_ohe = torch.from_numpy(y_valid).to(torch.float32)

In [ ]:
def cross_validation(estimator, scoring):
    _, train_scores, test_scores = learning_curve(estimator, X_train, y_train_ohe, cv=5, n_jobs=-1, train_sizes=[1.0, ], scoring=scoring)
    test_scores = test_scores[0]
    mean, std = test_scores.mean(), test_scores.std()
    return mean, std
def plot(title, xlabel, X_plot, Y_plot, Y_HEEE, error):
    X_new = np.array(X_plot)
    Y_new = np.array(Y_plot)
    Y_HEHEHE = np.array(Y_HEEE)
    errors = np.array(error)
    plt.xlabel(xlabel)
    plt.title(title)
    plt.ylabel("Point")
    plt.plot(X_new,Y_new, color = "black", label = "Train accuracy")
    plt.plot(X_new,Y_HEHEHE, color = "blue", label = "Validating accuracy")
    plt.fill_between(X_new,Y_new-errors, Y_new+errors, color = "gray")
    plt.legend()

In [ ]:
class CNN_2(nn.Module):
    def __init__(self,input_dim,output_dim,dropout,filter=16,num_hidden=20):
        super().__init__()
        self.conv1 = nn.Conv1d(1,filter,3, padding = 1)
        self.conv2 = nn.Conv1d(filter,filter,3, padding = 1)
        self.conv3 = nn.Conv1d(filter,filter,3, padding = 1)
        self.maxpool1 = nn.MaxPool1d(2) 
        self.maxpool2 = nn.MaxPool1d(3) 
        self.conv4 = nn.Conv1d(filter,filter,3, padding = 1)
        self.flatten = nn.Flatten()
        self.dense = nn.Linear(int(filter*input_dim/6),num_hidden)
        self.drop = nn.Dropout(dropout)
        self.dense2 = nn.Linear(num_hidden,output_dim)
        self.prelu = nn.PReLU()
        self.softmax = nn.Softmax(dim=1)
        
    def forward(self,X):
        X = self.conv1(X)
        X = self.prelu(X)
        # X = self.conv2(X)
        # X = self.prelu(X)
        X = self.maxpool1(X)
        X = self.conv3(X)
        X = self.prelu(X)
        # X = self.conv4(X)
        # X = self.prelu(X)
        X = self.maxpool2(X)
        X = self.flatten(X)
        X = self.drop(X)
        X = self.dense(X)
        X = self.prelu(X)
        X = self.dense2(X)
        X = self.prelu(X)
        X = self.softmax(X)
        return X

In [ ]:
optimizer = torch.optim.SGD
X_train.shape

In [ ]:
# How to choose the best hyper-parameters using cross-validation k-fold and save the figure

title = "CNN learning rate"
xlabel = "Learning rate"
X_plot = []
Y_plot = []
Y_HEEE = []

error = []

In [ ]:
for n in tqdm([0.01,0.02,0.03,0.05,0.1,0.2]):
    clf = NeuralNetClassifier(module = CNN,module__input_dim = 198,module__output_dim = 2, module__dropout = 0.3, max_epochs = 50,
                            lr = n, verbose = 0, optimizer = optimizer, device = "cuda" if torch.cuda.is_available() else "cpu",
                            criterion = nn.BCELoss, train_split=None) 
    mean, std = cross_validation(clf, "accuracy")
    clf.fit(X_train,y_train_ohe)
    y_pred_valid_ohe = clf.predict(X_valid)
    y_pred_valid = ohc.inverse_transform(y_pred_valid_ohe)
    X_plot.append(str(n))
    Y_plot.append(mean)
    error.append(std)
    Y_HEEE.append(accuracy_score(y_pred_valid,y_valid))

In [ ]:
plot(title, xlabel, X_plot, Y_plot, Y_HEEE, error)
plt.plot()
plt.savefig('images/cnn/cnn_learning_rate' + classify_type + ".png", bbox_inches='tight')

In [ ]:
# How to choose the best hyper-parameters using cross-validation k-fold and save the figure

title = "CNN dropout rate"
xlabel = "Dropout"
X_plot = []
Y_plot = []
Y_HEEE = []

error = []
for n in tqdm([0.1,0.2,0.3,0.4,0.5]):
    clf = NeuralNetClassifier(module = CNN,module__input_dim = 198, module__output_dim = 2, module__dropout = n, max_epochs = 50,
                            lr = 0.03, verbose = 0, optimizer = optimizer, device = "cuda" if torch.cuda.is_available() else "cpu",
                            criterion = nn.BCELoss, train_split=None) 
    mean, std = cross_validation(clf, "accuracy")
    clf.fit(X_train,y_train_ohe)
    y_pred_valid_ohe = clf.predict(X_valid)
    y_pred_valid = ohc.inverse_transform(y_pred_valid_ohe)
    X_plot.append(str(n))
    Y_plot.append(mean)
    error.append(std)
    Y_HEEE.append(accuracy_score(y_pred_valid,y_valid))
plot(title, xlabel, X_plot, Y_plot, Y_HEEE, error)
plt.plot()
plt.savefig('images/cnn/cnn_dropout' + classify_type + ".png", bbox_inches='tight')

In [ ]:
ada = NeuralNetClassifier(module = CNN_2,module__input_dim = 198,module__filter=48,module__num_hidden=20,
                          module__output_dim = 2, module__dropout = 0.4, max_epochs = 100,
                          lr = 0.03, verbose = 1, optimizer = optimizer, device = "cuda" if torch.cuda.is_available() else "cpu",
                          criterion = nn.BCELoss, train_split=None) 

In [ ]:
ada.fit(X_train,y_train_ohe)
y_predict_train_ohe=ada.predict(X_train)
y_predict_ohe=ada.predict(X_test)
y_pred_true_ohe = ada.predict(X_true)
y_predict_train = ohc.inverse_transform(y_predict_train_ohe)
y_predict = ohc.inverse_transform(y_predict_ohe)
y_pred_true = ohc.inverse_transform(y_pred_true_ohe)

In [ ]:
from sklearn.metrics import roc_auc_score
display_labels = {"binary":["Benign","Malware"],"multiclass":["benign","generic","virus","ransomware","rat","spyware","worm"]}
print("Train metrics :")
print("\tAccuracy:", accuracy_score(y_train, y_predict_train))
if classify_type == "binary":
    print("\tRecall:", recall_score(y_train, y_predict_train))
    print("\tF1:",f1_score(y_train, y_predict_train))
    print("\tAUC:", roc_auc_score(y_train, y_predict_train))
print("Test metrics :")
print("\tAccuracy:", accuracy_score(y_test, y_predict))
if classify_type == "binary":
    print("\tRecall:", recall_score(y_test, y_predict))
    print("\tF1:",f1_score(y_test, y_predict))
    print("\tAUC:", roc_auc_score(y_test, y_predict))
print("True Test metrics :")
print("\tAccuracy:", accuracy_score(y_true, y_pred_true))
if classify_type == "binary":
    print("\tRecall:", recall_score(y_true, y_pred_true))
    print("\tF1:",f1_score(y_true, y_pred_true))
    print("\tAUC:", roc_auc_score(y_true, y_pred_true))

In [ ]:
# Todo: Save the models you trained into models folder, using joblib/pickle.
# Only save the binary one.
import pickle
if classify_type == "binary":
    with open('models/cnn.pkl', 'wb') as f:
        pickle.dump(ada, f)